# Environment Check

This notebook verifies that both workshop environments are correctly installed.  
Run each section using the **matching conda kernel** (see headers below).

---

## Part 1 — Nowcasting Environment (`main`)

**Kernel to use:** `main`

**First-time setup** — open **Anaconda Prompt** and run each line:

```
conda create -n main python=3.11 pip numpy pandas matplotlib seaborn scikit-learn tqdm python-dateutil geopandas jupyterlab ipykernel joblib statsmodels openpyxl -y
conda activate main
conda clean --packages --tarballs
conda install pytorch torchvision torchaudio cpuonly -c pytorch
conda install pip pmdarima -y
pip install nowcast_lstm
pip install eurostat
```

**Expected output** (cell below):
```
LinearRegression OK — R² = 0.94XX
LSTM OK — predictions shape: (41, 3)
```


In [ ]:
# Utilities
import io, os, re, urllib3, requests, datetime, glob, json
import pandas as pd
import numpy as np
from datetime import date
from bs4 import BeautifulSoup
from dateutil.parser import parse
import xml.etree.ElementTree as ET
from io import BytesIO
import zipfile
import eurostat
import pprint as pp
import logging

# Statistics
import matplotlib.pyplot as plt
import pmdarima as pm
import torch
from nowcast_lstm.LSTM import LSTM
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ── LinearRegression smoke test ───────────────────────────────────────────────
rng = np.random.default_rng(42)
X_train = rng.standard_normal((100, 3))
y_train = X_train @ np.array([1.5, -2.0, 0.8]) + rng.standard_normal(100) * 0.5

X_test = rng.standard_normal((20, 3))
y_test = X_test @ np.array([1.5, -2.0, 0.8]) + rng.standard_normal(20) * 0.5

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"LinearRegression OK — R² = {r2:.4f}")
print(f"Coefficients: {model.coef_.round(3)}")


# ── LSTM smoke test ───────────────────────────────────────────────────────────
# Synthetic quarterly data: 40 quarters of training, 4 of test
rng = np.random.default_rng(0)
n_obs = 44
dates = pd.date_range("2010-01-01", periods=n_obs, freq="QS")
x1 = rng.standard_normal(n_obs)
x2 = rng.standard_normal(n_obs)
target = 1.5 * x1 - 2.0 * x2 + rng.standard_normal(n_obs) * 0.3

df = pd.DataFrame({"date": dates, "gdp": target, "x1": x1, "x2": x2})
train_lstm = df.iloc[:40].reset_index(drop=True)
full_lstm  = df.reset_index(drop=True)

lstm_params = {
    "n_timesteps"            : 4,
    "fill_na_func"           : np.nanmean,
    "fill_ragged_edges_func" : np.nanmean,
    "n_models"               : 2,
    "train_episodes"         : 10,
    "batch_size"             : 30,
    "decay"                  : 0.98,
    "n_hidden"               : 8,
    "n_layers"               : 1,
    "dropout"                : 0.0,
    "criterion"              : torch.nn.MSELoss(),
    "optimizer"              : torch.optim.Adam,
    "optimizer_parameters"   : {"lr": 1e-2, "weight_decay": 0.0},
}

lstm_model = LSTM(data=train_lstm, target_variable="gdp", **lstm_params)
lstm_model.train(quiet=True)
preds = lstm_model.predict(full_lstm)

print(f"LSTM OK — predictions shape: {preds.shape}")
print(preds.tail(4).to_string(index=False))



---

## Part 2 — GIS Environment (`geo`)

**Kernel to use:** `geo`

**First-time setup** — open **Anaconda Prompt** and run each line:

```
conda create -n geo python=3.11 pip numpy pandas matplotlib seaborn tqdm python-dateutil geopandas jupyterlab gdal rasterio folium mapclassify fiona shapely pyproj colorcet h5py libgdal openpyxl -y
conda activate geo
conda install -c conda-forge contextily -y
pip install blarkmarblepy
```

**NASA Earthdata account** (required to download satellite files):

1. Register at https://ladsweb.modaps.eosdis.nasa.gov/
2. Go to your profile → *Eulas* → accept **Meris EULA** and **Sentinel-3 EULA**
3. Copy your Bearer token and paste it into `api_key` in the cell below

**Expected output** (cell below):
```
Downloaded: VNP46A2.A2025293.h10v07.XXX.h5
h5py OK  — top-level keys: ['HDFEOS', 'HDFEOS INFORMATION']
gdal OK  — 10 subdatasets found
[map of Jamaica nighttime lights]
Geo environment OK
```


In [ ]:
#### Replace API Key with your own (https://ladsweb.modaps.eosdis.nasa.gov/)
api_key = ""


# ── Geo / NTL smoke test: download one h5 file ────────────────────────────────
import struct, os, sys, time, requests, glob, re, h5py
import numpy as np, numpy.ma as ma
from osgeo import gdal, ogr
import matplotlib.pyplot as plt
import statistics as stat
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import geopandas as gpd
import contextily as cx
import colorcet as cc
import rasterio
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping

from osgeo import gdal
print(gdal.__version__)
drivers = [gdal.GetDriver(i).ShortName for i in range(gdal.GetDriverCount())]
print("HDF5" in drivers)   # should be True
print("HDF5Image" in drivers)  # should be True

# Target: 2025-10-20, Jamaica tile (day 293 of 2025)
year, day, tile = 2025, 293, "h10v07"

out_dir    = Path(os.getcwd()).parent / "hfiles"
output_dir = out_dir / "output"
out_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

headers  = {"Authorization": f"Bearer {api_key}"}
base_url = "https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A2"

# List files for that day and pick the matching tile
resp      = requests.get(f"{base_url}/{year}/{day:03d}.json", headers=headers)
filenames = resp.json().get("content", [])
matching  = [f["downloadsLink"] for f in filenames if tile in f["downloadsLink"] and f["downloadsLink"].endswith(".h5")]

if not matching:
    print("No matching file found — check API key or date/tile.")
else:
    file_url = matching[0]
    filename = file_url.split("/")[-1]
    out_path = out_dir / filename

    if out_path.exists():
        print(f"Already exists: {filename}")
    else:
        r = requests.get(file_url, headers=headers, stream=True)
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {filename}")

    # Verify with h5py
    with h5py.File(out_path, "r") as hf:
        print(f"h5py OK  — top-level keys: {list(hf.keys())}")

    # Verify gdal
    hdf = gdal.Open(str(out_path), gdal.GA_ReadOnly)
    sds = hdf.GetSubDatasets()
    print(f"gdal OK  — {len(sds)} subdatasets found")
    hdf = None

    # ── Map the downloaded file ────────────────────────────────────────────────
    gdf     = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_JAM_shp.zip")
    gdf_jam = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")

    hdflayer    = gdal.Open(str(out_path), gdal.GA_ReadOnly)
    subhdflayer = hdflayer.GetSubDatasets()[2][0]   # layer 2 = DNB_At_Sensor_Radiance
    rlayer      = gdal.Open(subhdflayer, gdal.GA_ReadOnly)

    meta        = rlayer.GetMetadata_Dict()
    H, V        = int(meta["HorizontalTileNumber"]), int(meta["VerticalTileNumber"])
    west, north = (10 * H) - 180, 90 - (10 * V)
    east, south = west + 10, north - 10

    tmp  = str(output_dir / "_test_tmp.tif")
    opts = gdal.TranslateOptions(gdal.ParseCommandLine(
        f"-a_srs EPSG:4326 -a_ullr {west} {north} {east} {south}"
    ))
    gdal.Translate(tmp, rlayer, options=opts)
    rlayer = None

    shapes = [mapping(geom) for geom in gdf_jam.geometry]
    with rasterio.open(tmp) as src:
        out_image, out_transform = rio_mask(src, shapes, crop=True, nodata=np.nan, filled=True)
        nrows, ncols = out_image.shape[1], out_image.shape[2]
        lons = out_transform.c + (np.arange(ncols) + 0.5) * out_transform.a
        lats = out_transform.f + (np.arange(nrows) + 0.5) * out_transform.e

    try: os.remove(tmp)
    except: pass

    data = out_image[0].astype(float)
    data[data >= 65535] = np.nan
    data[data < 0]      = np.nan

    LON, LAT   = np.meshgrid(lons, lats)
    valid      = data[~np.isnan(data)]
    vmin, vmax = np.nanpercentile(valid, 2), np.nanpercentile(valid, 98)
    date_label = datetime.strptime(f"{year}-{day:03d}", "%Y-%j").strftime("%b %d, %Y")

    fig, ax = plt.subplots(figsize=(10, 6))
    pcm = ax.pcolormesh(LON, LAT, data, cmap=cc.cm.bmy, vmin=vmin, vmax=vmax, shading="auto")
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, crs="EPSG:4326")
    fig.colorbar(pcm, ax=ax, orientation="vertical", label="NTL Radiance (nW/cm²/sr)")
    ax.set_title(f"Nighttime Lights — Jamaica ({date_label})", fontsize=13, weight="bold")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    fig.text(0.01, 0.01, "Source: NASA Black Marble VNP46A2", fontsize=9)
    plt.tight_layout()
    plt.show()

    # ── Cleanup ───────────────────────────────────────────────────────────────
    #out_path.unlink()
    #print(f"Deleted: {filename}")
    print("Geo environment OK")
